# 01 — 原始数据下载与初步处理

本 Notebook 负责从 **Amazon Reviews 2023** 和 **Goodreads** 官方源下载原始数据，
流式解析 `.jsonl.gz` 文件并提取关键字段，保存为高效的 **Parquet** 中间文件。

## 数据源

| 数据集 | 来源 | 说明 |
|--------|------|------|
| Amazon Reviews 2023 (8领域) | [官网](https://amazon-reviews-2023.github.io/) | 6个训练领域 + 2个OOD评估领域 |
| Goodreads | [UCSD](https://mengtingwan.github.io/data/goodreads.html) | OOD评估领域 |

### Amazon 8 个领域
- **训练 (AmazonMix-6)**: Video_Games, Arts_Crafts_and_Sewing, Movies_and_TV, Home_and_Kitchen, Electronics, Tools_and_Home_Improvement
- **OOD 评估**: Sports_and_Outdoors, Baby_Products

## 输出文件
```
data/intermediate/
├── {Category}_interactions.parquet   # 用户-物品交互 (user_id, parent_asin, timestamp)
├── {Category}_items.parquet          # 物品元数据 (parent_asin, title)
├── Goodreads_interactions.parquet
└── Goodreads_items.parquet
```

## 0. 环境准备

In [1]:
import os
import sys
import json
import gzip
import csv
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# 项目根目录
PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'

# 创建必要的目录
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"原始数据缓存: {RAW_DIR}")
print(f"中间文件目录: {INTERMEDIATE_DIR}")

ModuleNotFoundError: No module named 'pandas'

## 1. 定义数据集配置

Amazon Reviews 2023 的下载 URL 遵循统一模式：
- 评论数据：`https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_2023/raw/review_categories/{Category}.jsonl.gz`
- 元数据：`https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_2023/raw/meta_categories/meta_{Category}.jsonl.gz`

In [ ]:
# Amazon Reviews 2023 数据集配置
AMAZON_BASE_URL = "https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_2023/raw"

AMAZON_CATEGORIES = {
    # 6 个训练领域
    "Video_Games": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Video_Games.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Video_Games.jsonl.gz"},
    "Arts_Crafts_and_Sewing": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Arts_Crafts_and_Sewing.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Arts_Crafts_and_Sewing.jsonl.gz"},
    "Movies_and_TV": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Movies_and_TV.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Movies_and_TV.jsonl.gz"},
    "Home_and_Kitchen": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Home_and_Kitchen.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Home_and_Kitchen.jsonl.gz"},
    "Electronics": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Electronics.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Electronics.jsonl.gz"},
    "Tools_and_Home_Improvement": {"role": "train", "review_url": f"{AMAZON_BASE_URL}/review_categories/Tools_and_Home_Improvement.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Tools_and_Home_Improvement.jsonl.gz"},
    # 2 个 OOD 评估领域
    "Sports_and_Outdoors": {"role": "ood_eval", "review_url": f"{AMAZON_BASE_URL}/review_categories/Sports_and_Outdoors.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Sports_and_Outdoors.jsonl.gz"},
    "Baby_Products": {"role": "ood_eval", "review_url": f"{AMAZON_BASE_URL}/review_categories/Baby_Products.jsonl.gz", "meta_url": f"{AMAZON_BASE_URL}/meta_categories/meta_Baby_Products.jsonl.gz"},
}

# Goodreads 数据集配置
GOODREADS_CONFIG = {
    "interactions_url": "https://datarepo.eng.ucsd.edu/mcauley_group/gdrive/goodreads/goodreads_interactions.csv",
    "book_id_map_url": "https://datarepo.eng.ucsd.edu/mcauley_group/gdrive/goodreads/book_id_map.csv",
    "books_url": "https://datarepo.eng.ucsd.edu/mcauley_group/gdrive/goodreads/goodreads_books.json.gz",
}

print(f"📦 Amazon 领域数量: {len(AMAZON_CATEGORIES)}")
for cat, info in AMAZON_CATEGORIES.items():
    print(f"  - {cat} ({info['role']})")
print(f"📚 Goodreads: 交互+书籍元数据")

## 2. 下载工具函数

使用流式下载，带进度条，支持断点续传。

In [ ]:
def download_file(url: str, save_path: Path, chunk_size: int = 8192) -> Path:
    """
    流式下载文件，带进度条，支持跳过已下载文件。
    
    Args:
        url: 下载链接
        save_path: 保存路径
        chunk_size: 每次读取的字节数
    Returns:
        保存路径
    """
    save_path = Path(save_path)
    
    # 如果文件已存在，跳过下载
    if save_path.exists() and save_path.stat().st_size > 0:
        size_mb = save_path.stat().st_size / (1024 * 1024)
        print(f"  ✅ 已存在，跳过下载: {save_path.name} ({size_mb:.1f} MB)")
        return save_path
    
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"  ⬇️  正在下载: {url}")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(save_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=save_path.name) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                pbar.update(len(chunk))
    
    size_mb = save_path.stat().st_size / (1024 * 1024)
    print(f"  ✅ 下载完成: {save_path.name} ({size_mb:.1f} MB)")
    return save_path

## 3. Amazon Reviews 2023 — 流式解析工具

原始评论文件可能非常大（如 Home_and_Kitchen 有 67.4M 评论），
所以我们采用**流式逐行读取** `.jsonl.gz`，只提取需要的字段：
- 评论数据 → `user_id`, `parent_asin`, `timestamp`
- 元数据 → `parent_asin`, `title`

In [ ]:
def parse_amazon_reviews_stream(gz_path: Path) -> pd.DataFrame:
    """
    流式解析 Amazon Reviews .jsonl.gz 文件，提取交互三元组。
    
    字段说明 (Amazon Reviews 2023 格式):
    - user_id: 用户唯一标识
    - parent_asin: 父产品 ASIN（多个子变体归到同一父产品）
    - timestamp: 评论时间戳（毫秒级 Unix 时间戳）
    
    Returns:
        DataFrame with columns: [user_id, parent_asin, timestamp]
    """
    records = []
    with gzip.open(gz_path, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"解析 {gz_path.name}"):
            try:
                obj = json.loads(line.strip())
                # 只保留有完整信息的记录
                if 'user_id' in obj and 'parent_asin' in obj and 'timestamp' in obj:
                    records.append({
                        'user_id': obj['user_id'],
                        'parent_asin': obj['parent_asin'],
                        'timestamp': obj['timestamp'],
                    })
            except (json.JSONDecodeError, KeyError):
                continue
    
    df = pd.DataFrame(records)
    print(f"  📊 解析完成: {len(df):,} 条交互记录, {df['user_id'].nunique():,} 用户, {df['parent_asin'].nunique():,} 物品")
    return df


def parse_amazon_metadata_stream(gz_path: Path) -> pd.DataFrame:
    """
    流式解析 Amazon 元数据 .jsonl.gz 文件，提取物品标题。
    
    Returns:
        DataFrame with columns: [parent_asin, title]
    """
    records = []
    with gzip.open(gz_path, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"解析 {gz_path.name}"):
            try:
                obj = json.loads(line.strip())
                if 'parent_asin' in obj and 'title' in obj:
                    title = obj['title']
                    # 清洗标题：去掉空白、换行
                    if title and isinstance(title, str):
                        title = title.strip().replace('\n', ' ').replace('\r', ' ')
                        if title:  # 确保不为空
                            records.append({
                                'parent_asin': obj['parent_asin'],
                                'title': title,
                            })
            except (json.JSONDecodeError, KeyError):
                continue
    
    df = pd.DataFrame(records)
    # 去重 (一个 parent_asin 可能出现多次)
    df = df.drop_duplicates(subset='parent_asin', keep='first')
    print(f"  📊 解析完成: {len(df):,} 个物品有标题")
    return df

## 4. 下载并解析全部 8 个 Amazon 领域

对每个领域：
1. 下载评论数据 `.jsonl.gz` → 流式解析提取 `(user_id, parent_asin, timestamp)`
2. 下载元数据 `.jsonl.gz` → 流式解析提取 `(parent_asin, title)`
3. 保存为 Parquet 中间文件

⚠️ **注意**：某些领域数据量很大，下载和解析可能需要较长时间。
建议先跑 `Video_Games`（最小的领域）验证流程。

In [ ]:
# 你可以修改这个列表来控制先处理哪些领域
# 建议先跑 Video_Games 验证流程，确认无误后再处理全部
#CATEGORIES_TO_PROCESS = list(AMAZON_CATEGORIES.keys())
CATEGORIES_TO_PROCESS = ["Video_Games"]  # 只处理一个领域用于测试

print(f"将处理 {len(CATEGORIES_TO_PROCESS)} 个 Amazon 领域:")
for cat in CATEGORIES_TO_PROCESS:
    print(f"  - {cat} ({AMAZON_CATEGORIES[cat]['role']})")

In [ ]:
amazon_stats = {}

for category in CATEGORIES_TO_PROCESS:
    config = AMAZON_CATEGORIES[category]
    print(f"\n{'='*60}")
    print(f"🔄 处理 Amazon 领域: {category} ({config['role']})")
    print(f"{'='*60}")
    
    # 检查是否已有中间文件
    interactions_parquet = INTERMEDIATE_DIR / f"{category}_interactions.parquet"
    items_parquet = INTERMEDIATE_DIR / f"{category}_items.parquet"
    
    if interactions_parquet.exists() and items_parquet.exists():
        print(f"  ✅ 中间文件已存在，跳过下载和解析")
        df_interactions = pd.read_parquet(interactions_parquet)
        df_items = pd.read_parquet(items_parquet)
    else:
        # Step 1: 下载
        print(f"\n  📥 Step 1: 下载原始数据")
        review_gz = download_file(config['review_url'], RAW_DIR / f"{category}.jsonl.gz")
        meta_gz = download_file(config['meta_url'], RAW_DIR / f"meta_{category}.jsonl.gz")
        
        # Step 2: 流式解析
        print(f"\n  📖 Step 2: 流式解析评论数据")
        df_interactions = parse_amazon_reviews_stream(review_gz)
        
        print(f"\n  📖 Step 3: 流式解析元数据")
        df_items = parse_amazon_metadata_stream(meta_gz)
        
        # Step 3: 保存中间文件
        print(f"\n  💾 Step 4: 保存 Parquet 中间文件")
        df_interactions.to_parquet(interactions_parquet, index=False)
        df_items.to_parquet(items_parquet, index=False)
        print(f"  ✅ 已保存: {interactions_parquet.name}, {items_parquet.name}")
    
    # 统计
    amazon_stats[category] = {
        'raw_interactions': len(df_interactions),
        'raw_users': df_interactions['user_id'].nunique(),
        'raw_items': df_interactions['parent_asin'].nunique(),
        'items_with_title': len(df_items),
    }
    print(f"\n  📊 统计: {amazon_stats[category]}")

In [ ]:
# 显示所有 Amazon 领域的原始数据统计
print("\n" + "="*80)
print("📊 Amazon Reviews 2023 — 原始数据统计汇总")
print("="*80)
print(f"{'领域':<35} {'交互数':>15} {'用户数':>12} {'物品数':>12} {'有标题':>12}")
print("-"*80)
for cat, stats in amazon_stats.items():
    role_tag = '🏋️' if AMAZON_CATEGORIES[cat]['role'] == 'train' else '🧪'
    print(f"{role_tag} {cat:<32} {stats['raw_interactions']:>15,} {stats['raw_users']:>12,} {stats['raw_items']:>12,} {stats['items_with_title']:>12,}")
print("-"*80)
print("🏋️ = 训练领域 (AmazonMix-6)  |  🧪 = OOD评估领域")

## 5. Goodreads 数据集下载与解析

Goodreads 数据集作为 **域外 (OOD)** 评估数据集，不参与训练。

需要下载：
1. `goodreads_interactions.csv` (~4.1GB) — 用户-书籍交互
2. `book_id_map.csv` — 书籍 CSV ID 到 book_id 的映射
3. `goodreads_books.json.gz` (~2GB) — 书籍元数据（提取标题）

### Goodreads 数据格式说明

**交互数据** (`goodreads_interactions.csv`):
- 列: `user_id, book_id, is_read, rating, is_reviewed`
- 我们需要 `user_id`, `book_id`, 并把行号作为时间戳代替（Goodreads 无时间戳）
- ⚠️ Goodreads 交互没有时间戳字段。通常做法是用交互顺序作为代理时间戳

**书籍 ID 映射** (`book_id_map.csv`):
- 列: `book_id_csv, book_id`
- 交互文件中的 book_id 实际上是 csv 索引（从0开始），需要映射到真正的 book_id

In [ ]:
print("="*60)
print("🔄 处理 Goodreads 数据集")
print("="*60)

goodreads_interactions_parquet = INTERMEDIATE_DIR / "Goodreads_interactions.parquet"
goodreads_items_parquet = INTERMEDIATE_DIR / "Goodreads_items.parquet"

if goodreads_interactions_parquet.exists() and goodreads_items_parquet.exists():
    print("  ✅ 中间文件已存在，跳过下载和解析")
    df_gr_interactions = pd.read_parquet(goodreads_interactions_parquet)
    df_gr_items = pd.read_parquet(goodreads_items_parquet)
else:
    # Step 1: 下载
    print("\n  📥 Step 1: 下载原始数据")
    gr_interactions_csv = download_file(GOODREADS_CONFIG['interactions_url'], RAW_DIR / 'goodreads_interactions.csv')
    gr_book_id_map_csv = download_file(GOODREADS_CONFIG['book_id_map_url'], RAW_DIR / 'book_id_map.csv')
    gr_books_gz = download_file(GOODREADS_CONFIG['books_url'], RAW_DIR / 'goodreads_books.json.gz')
    
    # Step 2: 解析书籍元数据，提取标题
    print("\n  📖 Step 2: 解析书籍元数据 (提取标题)")
    book_titles = {}  # book_id -> title
    with gzip.open(gr_books_gz, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="解析 goodreads_books.json.gz"):
            try:
                obj = json.loads(line.strip())
                bid = obj.get('book_id', '')
                title = obj.get('title', '')
                if bid and title and isinstance(title, str):
                    title = title.strip().replace('\n', ' ').replace('\r', ' ')
                    if title:
                        book_titles[str(bid)] = title
            except (json.JSONDecodeError, KeyError):
                continue
    print(f"  📊 获取 {len(book_titles):,} 本书的标题")
    
    # Step 3: 解析 book_id_map
    print("\n  📖 Step 3: 解析 book_id_map.csv")
    df_book_id_map = pd.read_csv(gr_book_id_map_csv)
    print(f"  📊 映射表大小: {len(df_book_id_map):,}")
    print(f"  列名: {list(df_book_id_map.columns)}")
    print(df_book_id_map.head())
    # 构建 csv_index -> book_id 的映射
    csv_to_bookid = dict(zip(df_book_id_map['book_id_csv'].astype(str), df_book_id_map['book_id'].astype(str)))
    
    # Step 4: 解析交互数据 (分块读取大文件)
    print("\n  📖 Step 4: 解析交互数据 (分块读取)")
    interaction_records = []
    chunk_size = 1_000_000
    row_counter = 0
    
    for chunk in tqdm(pd.read_csv(gr_interactions_csv, chunksize=chunk_size), desc="读取 goodreads_interactions.csv"):
        for _, row in chunk.iterrows():
            user_id = str(row['user_id'])
            book_csv_id = str(row['book_id'])
            is_read = row.get('is_read', 0)
            
            # 只保留已读的交互
            if is_read:
                book_id = csv_to_bookid.get(book_csv_id, None)
                if book_id and book_id in book_titles:
                    interaction_records.append({
                        'user_id': user_id,
                        'book_id': book_id,
                        'timestamp': row_counter,  # 用行号作为时间代理
                    })
            row_counter += 1
    
    df_gr_interactions = pd.DataFrame(interaction_records)
    print(f"  📊 有效交互: {len(df_gr_interactions):,}, 用户: {df_gr_interactions['user_id'].nunique():,}, 书籍: {df_gr_interactions['book_id'].nunique():,}")
    
    # 构建物品 DataFrame
    unique_books = df_gr_interactions['book_id'].unique()
    df_gr_items = pd.DataFrame([
        {'book_id': bid, 'title': book_titles[bid]} 
        for bid in unique_books if bid in book_titles
    ])
    print(f"  📊 有交互且有标题的书籍: {len(df_gr_items):,}")
    
    # Step 5: 保存中间文件
    print("\n  💾 Step 5: 保存 Parquet 中间文件")
    df_gr_interactions.to_parquet(goodreads_interactions_parquet, index=False)
    df_gr_items.to_parquet(goodreads_items_parquet, index=False)
    print(f"  ✅ 已保存: {goodreads_interactions_parquet.name}, {goodreads_items_parquet.name}")

print(f"\n📊 Goodreads 统计:")
print(f"  交互数: {len(df_gr_interactions):,}")
print(f"  用户数: {df_gr_interactions['user_id'].nunique():,}")
print(f"  书籍数: {df_gr_interactions['book_id'].nunique():,}")
print(f"  有标题的书籍: {len(df_gr_items):,}")

## 6. 数据预览

让我们检查一下下载的数据长什么样。

In [ ]:
# 预览 Amazon 数据 (以 Video_Games 为例)
print("🎮 Amazon Video_Games 交互数据预览:")
sample_interactions = pd.read_parquet(INTERMEDIATE_DIR / "Video_Games_interactions.parquet")
print(sample_interactions.head(10))
print(f"\n数据类型:\n{sample_interactions.dtypes}")
print(f"\ntimestamp 范围: {sample_interactions['timestamp'].min()} ~ {sample_interactions['timestamp'].max()}")

print("\n" + "="*60)
print("🎮 Amazon Video_Games 物品元数据预览:")
sample_items = pd.read_parquet(INTERMEDIATE_DIR / "Video_Games_items.parquet")
print(sample_items.head(10))
print(f"\n标题长度统计:")
print(sample_items['title'].str.len().describe())

In [ ]:
# 预览 Goodreads 数据
print("📚 Goodreads 交互数据预览:")
print(df_gr_interactions.head(10))

print("\n📚 Goodreads 书籍元数据预览:")
print(df_gr_items.head(10))

## 7. 可视化: 原始数据规模对比

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# 准备数据
all_stats = {}
for cat, stats in amazon_stats.items():
    all_stats[cat] = stats
all_stats['Goodreads'] = {
    'raw_interactions': len(df_gr_interactions),
    'raw_users': df_gr_interactions['user_id'].nunique(),
    'raw_items': df_gr_interactions['book_id'].nunique(),
}

categories = list(all_stats.keys())
interactions = [all_stats[c]['raw_interactions'] for c in categories]
users = [all_stats[c]['raw_users'] for c in categories]
items = [all_stats[c]['raw_items'] for c in categories]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 颜色：训练领域蓝色，OOD领域红色，Goodreads绿色
colors = []
for c in categories:
    if c == 'Goodreads':
        colors.append('#2ecc71')
    elif AMAZON_CATEGORIES.get(c, {}).get('role') == 'ood_eval':
        colors.append('#e74c3c')
    else:
        colors.append('#3498db')

short_names = [c.replace('_and_', '\n&\n').replace('_', '\n') for c in categories]

for ax, data, title in zip(axes, [interactions, users, items], ['交互数', '用户数', '物品数']):
    bars = ax.bar(range(len(categories)), data, color=colors)
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'原始{title}', fontsize=14)
    ax.set_ylabel(title)
    # 在柱子上方显示数值
    for bar, val in zip(bars, data):
        if val >= 1_000_000:
            label = f"{val/1_000_000:.1f}M"
        elif val >= 1_000:
            label = f"{val/1_000:.0f}K"
        else:
            label = str(val)
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), label,
                ha='center', va='bottom', fontsize=8)

plt.suptitle('LLM2Rec 原始数据规模对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
print("🔵 蓝色=训练领域 | 🔴 红色=OOD评估 | 🟢 绿色=Goodreads")

## 8. 中间文件清单检查

确认所有中间文件都已生成。

In [ ]:
print("📋 中间文件清单检查:")
print("="*60)

expected_files = []
for cat in AMAZON_CATEGORIES:
    expected_files.append(INTERMEDIATE_DIR / f"{cat}_interactions.parquet")
    expected_files.append(INTERMEDIATE_DIR / f"{cat}_items.parquet")
expected_files.append(INTERMEDIATE_DIR / "Goodreads_interactions.parquet")
expected_files.append(INTERMEDIATE_DIR / "Goodreads_items.parquet")

all_ok = True
for fp in expected_files:
    exists = fp.exists()
    if exists:
        size_mb = fp.stat().st_size / (1024 * 1024)
        print(f"  ✅ {fp.name:45s} ({size_mb:>8.1f} MB)")
    else:
        print(f"  ❌ {fp.name:45s} (缺失!)")
        all_ok = False

print(f"\n{'✅ 所有文件就绪，可以进入 02_filter_and_split.ipynb' if all_ok else '❌ 有文件缺失，请检查上述步骤'}")

## 📝 小结

本 Notebook 完成了以下工作：

1. ✅ 从 Amazon Reviews 2023 官方源下载 8 个领域的评论数据和元数据
2. ✅ 从 Goodreads 下载交互数据和书籍元数据
3. ✅ 流式解析所有 `.jsonl.gz` 文件，提取关键字段
4. ✅ 保存为高效的 Parquet 中间文件

### 下一步
→ 打开 `02_filter_and_split.ipynb`，进行 5-core 过滤、序列构建和数据划分。